In [1]:
import news_lib as nl
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "stock_crawler"))
import stock_lib as sl
from datetime import date, timedelta
import pandas as pd

In [2]:
df_stock = sl.get_stock("NVDA",date(2026, 6, 1), date(2026, 7, 1))
df_stock.head()

,symbol,date,open,close,high,low,volume,dividends,stock_split
0,NVDA,2026-06-01,215.478858,224.098816,224.608217,215.448894,212850700,0.00,0.0
1,NVDA,2026-06-02,226.915518,222.560608,232.009586,221.092318,193362900,0.00,0.0
2,NVDA,2026-06-03,221.461887,214.500000,222.560613,214.260274,160907000,0.00,0.0
3,NVDA,2026-06-04,213.910004,218.660004,221.600006,210.970001,169022200,0.25,0.0
4,NVDA,2026-06-05,214.529999,205.100006,214.869995,204.330002,219655500,0.00,0.0


In [3]:
df_sig_dates = sl.spot_significant_dates(df_stock, "close")
df_sig_dates.head()

,symbol,date,open,close,high,low,volume,dividends,stock_split,percent_change,percent_increase
4,NVDA,2026-06-05,214.529999,205.100006,214.869995,204.330002,219655500,0.0,0.0,0.062014,False
15,NVDA,2026-06-23,202.169998,200.039993,203.770004,200.000000,153496200,0.0,0.0,0.041265,False
7,NVDA,2026-06-10,204.429993,200.419998,207.220001,199.919998,161746600,0.0,0.0,0.037322,False


In [ ]:
# parameters
day_range = 2
FINANCE_PUBLISHERS = ["finance.yahoo.com", "benzinga.com"]
# function workflow
news_arr = []
for i, v in enumerate(df_sig_dates.itertuples()):
    start_date = (v.date - timedelta(days=day_range)).date()
    end_date = (v.date + timedelta(days=day_range)).date()
    symbol = v.symbol
    # check db for the news
    news = nl.get_articles(symbol, start_date, end_date)
    if len(news) < 1:
        print("CRAWL")
        # CRAWL NEW NEWS
        # build RSS query
        query = nl.query_builder(symbol, start_date, end_date, FINANCE_PUBLISHERS)
        # fetch
        fetched_news = nl.news_filter(query, start_date, end_date, FINANCE_PUBLISHERS, limit=3)
        # CRAWL FULL TEXTS
        complete_news = nl.crawl_full_text(fetched_news)
        # UPSERT INTO DB
        nl.upsert_articles(symbol, complete_news)
        # GET THE ARTICLE
        news = nl.get_articles(symbol, start_date, end_date)
    else:
        print("ALL GOOD") 
    news_arr.append(news)
# ATTACH CHANGES TO THE ARTICLES
flat_news = [article for sublist in news_arr for article in sublist]
news_df = pd.DataFrame(flat_news)
news_df = news_df.drop_duplicates(subset="url")


ALL GOOD
ALL GOOD
ALL GOOD


In [5]:
for i in news_df.columns:
    print(f"{i}: {news_df[i].dtype}")

url: object
symbol: object
title: object
date: object
source: object
summary: object
full_text: object


In [14]:
df_sig_dates["date"].dtype

dtype('<M8[ns]')

In [6]:
# params
df_dates = df_sig_dates.copy()
df_news = news_df.copy()
windows = 2
#
df_news["date"] = pd.to_datetime(df_news["date"])
df_dates['date'] = pd.to_datetime(df_dates['date'], errors='coerce')
df_dates = df_dates.set_index('date')

def get_linked_info(article_date):
    window_start = article_date - pd.Timedelta(days=windows)
    window_end = article_date + pd.Timedelta(days=windows)
    in_window = df_dates[(df_dates.index >= window_start) & (df_dates.index <= window_end)]
    return [
        {
            "date": str(d.date()),
            "percent_change": row["percent_change"] * (-1 if not row["percent_increase"] else 1),
            "days_from_publish": (article_date - d).days,
        }
        for d, row in in_window.iterrows()
    ]
df_news["linked_info"] = df_news["date"].apply(get_linked_info)

In [15]:
for i in df_news.columns:
    print(f"{i}: {df_news[i].dtype}")

url: object
symbol: object
title: object
date: datetime64[ns]
source: object
summary: object
full_text: object
linked_info: object


In [76]:
df_news["linked_info"][3]

[{'date': '2026-06-23',
  'percent_change': 0.04126528091164561,
  'increase_change': False,
  'days_from_publish': -1}]